# Data Understanding and Cleaning

**Objective:** Inspect raw dataset, resolve structural issues, correct data types, and handle missing/impossible values to generate a clean, ML-ready interim dataset.

In [11]:
import pandas as pd
import numpy as np

RAW_DATA_PATH = '../data/raw/used_cars_dataset_v2.csv'
df_raw = pd.read_csv(RAW_DATA_PATH)

print(f"Initial raw dataset shape: {df_raw.shape}")

Initial raw dataset shape: (14993, 11)


## 1. Deduplication
Investigating and removing exact row-for-row duplicates, which are assumed to be web-scraping artifacts.

In [12]:
duplicates_mask = df_raw.duplicated(keep=False)
duplicate_rows = df_raw[duplicates_mask]

# Sort by a few key columns so the duplicate pairs sit right next to each other in the output
# We'll use a mix of categorical and numerical columns to sort
cols_to_sort = list(df_raw.columns)
duplicate_rows_sorted = duplicate_rows.sort_values(by=cols_to_sort)

print(f"Total rows flagged as part of a duplicate group: {len(duplicate_rows)}")
# Display the first 6 rows to see 3 pairs of duplicates
display(duplicate_rows_sorted.head(6))

Total rows flagged as part of a duplicate group: 1731


,Brand,model,Year,Age,kmDriven,Transmission,Owner,FuelType,PostedDate,AdditionInfo,AskPrice
10952,Ambassador,Ambassador,2023,1,"54,000 km",Manual,first,Petrol,Nov-24,Ambassador Grand Petrol,"₹ 4,21,000"
14087,Ambassador,Ambassador,2023,1,"54,000 km",Manual,first,Petrol,Nov-24,Ambassador Grand Petrol,"₹ 4,21,000"
23,Audi,A3,2015,9,"42,000 km",Automatic,second,Diesel,Nov-24,"Audi A3 2.0 35 TDI Attraction, 2015, Diesel","₹ 10,50,000"
146,Audi,A3,2015,9,"42,000 km",Automatic,second,Diesel,Nov-24,"Audi A3 2.0 35 TDI Attraction, 2015, Diesel","₹ 10,50,000"
199,Audi,A3,2015,9,"42,000 km",Automatic,second,Diesel,Nov-24,"Audi A3 2.0 35 TDI Attraction, 2015, Diesel","₹ 10,50,000"
378,Audi,A3,2015,9,"42,000 km",Automatic,second,Diesel,Nov-24,"Audi A3 2.0 35 TDI Attraction, 2015, Diesel","₹ 10,50,000"


In [13]:
# Drop exact duplicates to prevent data leakage during train/test splits
df_clean = df_raw.drop_duplicates(keep='first').copy()

print(f"Rows before: {df_raw.shape[0]}")
print(f"Rows after dropping duplicates: {df_clean.shape[0]}")
print(f"Duplicates removed: {df_raw.shape[0] - df_clean.shape[0]}")

Rows before: 14993
Rows after dropping duplicates: 13996
Duplicates removed: 997


## 2. Data Type Correction
Numerical columns (`AskPrice`, `kmDriven`) were loaded as object types due to appended text (e.g., 'km', currency symbols) and formatting commas. These require regex stripping and numeric casting.

In [15]:
# inspect column datatypes
print("Column Datatypes:")
display(df_clean.dtypes)

Column Datatypes:


Brand             str
model             str
Year            int64
Age             int64
kmDriven          str
Transmission      str
Owner             str
FuelType          str
PostedDate        str
AdditionInfo      str
AskPrice          str
dtype: object

In [16]:
# Strip non-numeric characters and cast to appropriate numeric types
df_clean['AskPrice'] = df_clean['AskPrice'].astype(str).str.replace(r'\D', '', regex=True)
df_clean['AskPrice'] = pd.to_numeric(df_clean['AskPrice'], errors='coerce').astype('Int64')

df_clean['kmDriven'] = df_clean['kmDriven'].astype(str).str.replace(r'\D', '', regex=True)
df_clean['kmDriven'] = pd.to_numeric(df_clean['kmDriven'], errors='coerce') # Float to support NaNs

## 3. Implicit Missing Data Handling
Identifying extreme outliers or default system values that represent data entry errors rather than valid physical measurements.

In [17]:
# Generate summary statistics for numerical columns
print("Summary Statistics for Numerical Columns:")
display(df_clean.describe().round(2))

Summary Statistics for Numerical Columns:


,Year,Age,kmDriven,AskPrice
count,13996.00,13996.00,13909.0,13996.0
mean,2016.33,7.67,104871.9,979935.13
std,4.39,4.39,192501.8,1585757.6
min,1900.00,0.00,0.0,15000.0
25%,2014.00,5.00,46000.0,340000.0
50%,2017.00,7.00,69000.0,560000.0
75%,2019.00,10.00,92000.0,990000.0
max,2024.00,124.00,8000000.0,42500000.0


In [7]:
# 1. Investigate the 1900 Year cars
print("--- Cars from 1900 ---")
display(df_clean[df_clean['Year'] == 1900])

# 2. Investigate absurdly high kilometers (e.g., > 600,000 km)
print("\n--- Cars with > 600k km ---")
display(df_clean[df_clean['kmDriven'] > 600000])

# 3. Investigate the 4.25 Crore car
print("\n--- Most Expensive Car ---")
display(df_clean[df_clean['AskPrice'] == df_clean['AskPrice'].max()])

# 4. Investigate 0 km cars
print(f"\nTotal cars with exactly 0 km: {len(df_clean[df_clean['kmDriven'] == 0])}")

--- Cars from 1900 ---


,Brand,model,Year,Age,kmDriven,Transmission,Owner,FuelType,PostedDate,AdditionInfo,AskPrice
11392,Renault,Triber,1900,124,40000.0,Automatic,second,hybrid,Dec-24,Renault Triber 11 12 2019 CNG & Hybrids 40000 ...,440000



--- Cars with > 600k km ---


,Brand,model,Year,Age,kmDriven,Transmission,Owner,FuelType,PostedDate,AdditionInfo,AskPrice
1,Toyota,Innova,2009,15,1900000.0,Manual,second,Diesel,Jul-24,"Toyota Innova 2.5 G (Diesel) 7 Seater, 2009, D...",375000
22,Honda,City,2010,14,990000.0,Manual,first,Petrol,Nov-24,"Honda City S MT, 2010, Petrol",295000
44,Mercedes-Benz,C-Class,2015,9,650000.0,Automatic,first,Diesel,Nov-24,"Mercedes-Benz C-Class 2.0 220d Progressive, 20...",1750000
75,Toyota,Innova Crysta,2017,7,790000.0,Automatic,second,Diesel,Nov-24,"Toyota Innova Crysta 2.8 GX AT, 2017, Diesel",1725000
94,Volkswagen,Vento,2011,13,1340000.0,Manual,first,Diesel,Nov-24,"Volkswagen Vento 2010-2013 Diesel Highline, 20...",425000
...,...,...,...,...,...,...,...,...,...,...,...
14231,Maruti Suzuki,Estilo,2010,14,712000.0,Manual,first,hybrid,Nov-24,"Maruti Suzuki Estilo VXi, 2010, CNG & Hybrids",125000
14589,Mercedes-Benz,GL-Class,2015,9,1150000.0,Automatic,second,Diesel,Dec-24,"Mercedes-Benz GL-Class 3.0 350 CDI 4Matic, 201...",2490000
14911,Honda,Amaze,2018,6,950000.0,Manual,first,hybrid,Nov-24,Honda amaze 2018 cng Bilkul ok kondition gadi ...,320000
14979,Hyundai,Eon,2012,12,781313.0,Manual,second,Petrol,Nov-24,"Hyundai EON Magna +, 2012, Petrol",155000



--- Most Expensive Car ---


,Brand,model,Year,Age,kmDriven,Transmission,Owner,FuelType,PostedDate,AdditionInfo,AskPrice
3331,Rolls-Royce,Phantom Series II,2015,9,11500.0,Automatic,second,Petrol,Aug-24,"Rolls-Royce Phantom Series II EWB, 2015, Petrol",42500000



Total cars with exactly 0 km: 39


In [18]:
# Remove obvious system default placeholder (Year 1900)
df_clean = df_clean[df_clean['Year'] != 1900]

# Convert physically improbable kmDriven values to NaN for unified handling
df_clean.loc[(df_clean['kmDriven'] == 0) | (df_clean['kmDriven'] > 800000), 'kmDriven'] = np.nan

## 4. Explicit Missing Data Handling & Export
Handling remaining NaN values and exporting the finalized interim dataset for EDA.

In [19]:
# Missing values constitute ~2.4% of the dataset. 
# Dropping them is preferred to maintain pipeline simplicity and avoid imputation leakage.
df_clean = df_clean.dropna(subset=['kmDriven'])

INTERIM_DATA_PATH = '../data/interim/used_cars_cleaned.csv'
df_clean.to_csv(INTERIM_DATA_PATH, index=False)

print(f"Final clean dataset shape: {df_clean.shape}")
print(f"Exported to: {INTERIM_DATA_PATH}")

Final clean dataset shape: (13652, 11)
Exported to: ../data/interim/used_cars_cleaned.csv
